# Train the E7 RTA Helper pick model on a Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Step 2 asks for `colab_bundle.zip` (match data, hero types and `get_rec_model.py`).
3. Keep this tab open while it trains; Colab disconnects idle or very long sessions.
4. The last cell downloads `rec_model.h5` and `rec_variables.pkl`. Put both in the app's `data/` folder.

Training runs in its own Python 3.11 environment pinned to TensorFlow 2.17 / tf_keras / scikit-learn 1.5.1, the versions whose saved files the app (TensorFlow 2.13) can load.

In [ ]:
# 1. Check the GPU and build the training environment (a few minutes)
# Colab's own Python is too new for TensorFlow 2.17, so uv downloads Python 3.11 for this environment
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
!rm -rf /content/train-env
!uv venv -q --python 3.11 /content/train-env
!uv pip install -q --python /content/train-env/bin/python "tensorflow[and-cuda]==2.17.1" tf_keras==2.17.0 attention==5.0.0 scikit-learn==1.5.1 pandas==2.2.2 "numpy<2"
!/content/train-env/bin/python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))"

In [ ]:
# 2. Upload colab_bundle.zip
from google.colab import files
uploaded = files.upload()
# Save it to a fixed path, whatever folder the notebook is currently in
with open('/content/colab_bundle.zip', 'wb') as f:
    f.write(next(iter(uploaded.values())))
!rm -rf /content/e7 && mkdir -p /content/e7 && unzip -q -o /content/colab_bundle.zip -d /content/e7 && ls -R /content/e7

In [ ]:
# 3. Train (one line per epoch; stops by itself when validation stops improving)
%cd /content/e7
!TF_USE_LEGACY_KERAS=1 TF_CPP_MIN_LOG_LEVEL=2 /content/train-env/bin/python -u workflow_scripts/get_rec_model.py

In [ ]:
# 4. Download the trained model (put both files in the app's data/ folder)
from google.colab import files
files.download('/content/e7/data/rec_model.h5')
files.download('/content/e7/data/rec_variables.pkl')